In [52]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3
import emoji

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [15]:
def get_html(url = 'https://www.sindipetroamazonia.org.br/category/noticias/'):
    payload = {}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [16]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('time', class_='onDate date published')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for article in articles:
        a = article.find('a')
        link = a.get('href')
        date = article.get('datetime')
        date = date.split('T')
        date = datetime.strptime(date[0], "%Y-%m-%d")
        link_date = [link, date]
        news_links.append(link_date)

    return news_links

In [40]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [35]:
def get_next_page(fnp_url, next_page_number = 1):
    validated_news_links = []
    url = fnp_url + 'page/' + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [ ]:
def sanitize_paragraphs(paragraphs:list, min_paragraph_len = 500, concat_trigger_size = 1500):
    '''
    sanitiza os parágrafos, realizando replace de partes de strings e concatenando parágrafos
    
    paragraphs: lista de parágrafos
    min_paragraph_len: define quais parágrafos devem ser concatenados
    concat_trigger_size: caso o tamanho da concatenação de textos esteja acima desse valor, 
                            escreve como parágrafo e reseta a variável que armazena as 
                            strings a serem concatenadas
    '''
    def append_paragraphs(current_new_paragraph):
        new_paragraphs.append(current_new_paragraph.strip())
    
    paragraphs = [emoji.demojize(paragraph) for paragraph in [paragraph\
                                                              .replace('\xa0',' ')\
                                                              .replace('\n',' ')\
                                                              .replace('\t',' ')\
                                                              .replace('[email-protected]', '')\
                                                              .strip() \
                                                              for paragraph in paragraphs] \
                  if len(paragraph)>0] #removendo emoji, tratanto texto e realizando strip para então construir lista de parágrafos que tenham len > 0
    paragraphs_mask = [len(paragraph) < min_paragraph_len for paragraph in paragraphs] # true são os abaixo do min_paragraph_len, precisarao ser tratados
    if any(paragraphs_mask): #caso algum elemento precise ser tratado, trigga processo
        new_paragraphs = []
        current_new_paragraph = ''
        for i in range(len(paragraphs)):
            if len(current_new_paragraph) >= concat_trigger_size:
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
            if paragraphs_mask[i]: #caso seja menor que o min_paragraph_len
                current_new_paragraph = current_new_paragraph + ' ' + paragraphs[i]
                if i == len(paragraphs)-1: #caso seja o ultimo elemento
                    append_paragraphs(current_new_paragraph)
            else:
                current_new_paragraph = current_new_paragraph + '' + paragraphs[i]
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
                    
    return new_paragraphs

In [ ]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1', class_='entry-title').text
    paragraphs = soup.find('div', class_='entry-content')

    paragraphs = paragraphs.text.split('\n')
    paragraphs = sanitize_paragraphs(paragraphs)

    return title, paragraphs

In [39]:
next_page_number = 1
validated_news_links = []
url_default = 'https://www.sindipetroamazonia.org.br/category/noticias/'
url = url_default + 'page/' + str(next_page_number) + '/'
html_content = get_html(url)
news_links = get_links_and_dates(html_content)
validated_links, next_page = get_validated_links(news_links)

for validated_link in validated_links:
    validated_news_links.append(validated_link)

if next_page:
    validated_links = get_next_page(url_default, next_page_number + 1)
    
    for validated_link in validated_links:
        validated_news_links.append(validated_link)

len(validated_news_links)

25

In [64]:
def main():
    next_page_number = 1
    validated_news_links = []
    url_default = 'https://www.sindipetroamazonia.org.br/category/noticias/'
    url = url_default + 'page/' + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(url_default, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'PA_AM_MA_AP',
                    'url' : url,
                    'titulo' : title,
                    'data': date,
                    'paragrafo' : paragraph,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [68]:
result = main()
result

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.67s/it]


[{'sindicato': 'PA_AM_MA_AP',
  'url': 'https://www.sindipetroamazonia.org.br/2025/08/18/sindicato-exige-solucoes-da-petrobras-sobre-problemas-na-unidade-da-amazonia/',
  'titulo': 'Sindicato exige soluções da Petrobras sobre problemas na Unidade da Amazônia',
  'data': datetime.datetime(2025, 8, 18, 0, 0),
  'paragrafo': '',
  'num_paragrafo': 1},
 {'sindicato': 'PA_AM_MA_AP',
  'url': 'https://www.sindipetroamazonia.org.br/2025/08/18/sindicato-exige-solucoes-da-petrobras-sobre-problemas-na-unidade-da-amazonia/',
  'titulo': 'Sindicato exige soluções da Petrobras sobre problemas na Unidade da Amazônia',
  'data': datetime.datetime(2025, 8, 18, 0, 0),
  'paragrafo': 'A diretoria do Sindipetro Amazônia esteve reunida na última quinta-feira (14/08) com a gestão da Petrobras para cobrar soluções para uma extensa pauta de problemas que vêm gerando grande insatisfação e colocando em risco a segurança e o bem-estar dos trabalhadores.Banco de Horas: “Apropriação Indébita” do Tempo do Trabalha

In [66]:
len(result)

19

['',
 'A diretoria do Sindipetro Amazônia esteve reunida na última quinta-feira (14/08) com a gestão da Petrobras para cobrar soluções para uma extensa pauta de problemas que vêm gerando grande insatisfação e colocando em risco a segurança e o bem-estar dos trabalhadores.Banco de Horas: “Apropriação Indébita” do Tempo do Trabalhador',
 'Um dos pontos mais críticos da reunião foi a gestão do banco de horas. O sindicato denunciou a prática da Petrobras de obrigar os trabalhadores a tirar folgas para zerar seus saldos como uma “apropriação indébita” do tempo que pertence ao empregado. Essa medida unilateral não apenas desrespeita o acordo coletivo, que prevê escala definida, mas também causa prejuízos financeiros e logísticos aos trabalhadores.',
 'Em resposta, a gestão da Petrobras defendeu a medida, alegando ser uma orientação corporativa para controlar horas extras e garantir a viabilidade de manutenções importantes, e confirmou que o plano de gerenciamento será mantido.',
 'Diante da 

In [62]:
result = main()

df = pd.DataFrame(result)

df2 = df.explode('paragrafo')
df2.to_dict('records')

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.82s/it]


KeyError: 'paragrafo'

In [51]:
!pip install emoji

   ---------------------------------------- 0.0/590.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/590.6 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/590.6 kB ? eta -:--:--
   ---------------------------------------- 590.6/590.6 kB 2.0 MB/s  0:00:00
